In [4]:
import os
import random

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm


class FundusVesselExtractor:
    def __init__(
        self,
        input_green_dir="preprocessed_green_images",
        input_mask_dir="fundus_masks",
        output_mask_dir="vessel_masks",
        output_skeleton_dir="vessel_skeletons",
        debug_dir="vessel_debug_samples",
        metrics_csv="vessel_extraction_metrics.csv",
        debug_sample_count=1,
        random_seed=42,
        grid_size=8,

        process_scale=0.85,
        effective_mask_erode_size=15,

        frangi_sigmas=(0.8, 1.0, 1.4, 2.0, 2.8),
        frangi_beta=0.40,
        frangi_gamma_min=2.0,

        use_gabor=True,
        gabor_ksize=17,
        gabor_sigmas=(2.0, 3.0),
        gabor_lambdas=(8.0, 12.0),
        gabor_gamma=0.45,
        gabor_angle_step=30,

        frangi_weight=0.65,
        gabor_weight=0.35,
        product_keep_ratio=0.45,

        response_cap_percentile=99.5,
        response_median_ksize=3,

        dark_background_sigma=9,

        high_percentile=86.0,
        low_ratio=0.38,
        min_valid_response=3,

        weak_connection_dilate_size=11,
        strong_connection_dilate_size=7,

        min_expected_vessel_area_ratio=0.002,
        min_expected_skeleton_ratio=0.0005,

        remove_image_border_components=True,
        image_border_margin=4,

        use_skeleton=True,
        save_debug=True,
    ):
        self.input_green_dir = input_green_dir
        self.input_mask_dir = input_mask_dir
        self.output_mask_dir = output_mask_dir
        self.output_skeleton_dir = output_skeleton_dir
        self.debug_dir = debug_dir
        self.metrics_csv = metrics_csv
        self.debug_sample_count = debug_sample_count
        self.random_seed = random_seed
        self.grid_size = grid_size

        self.process_scale = process_scale
        self.effective_mask_erode_size = effective_mask_erode_size

        self.frangi_sigmas = frangi_sigmas
        self.frangi_beta = frangi_beta
        self.frangi_gamma_min = frangi_gamma_min

        self.use_gabor = use_gabor
        self.gabor_ksize = gabor_ksize
        self.gabor_sigmas = gabor_sigmas
        self.gabor_lambdas = gabor_lambdas
        self.gabor_gamma = gabor_gamma
        self.gabor_angle_step = gabor_angle_step

        # 重要: 前回エラーの原因。self代入を必ず行う。
        self.frangi_weight = frangi_weight
        self.gabor_weight = gabor_weight
        self.product_keep_ratio = product_keep_ratio

        self.response_cap_percentile = response_cap_percentile
        self.response_median_ksize = response_median_ksize

        self.dark_background_sigma = dark_background_sigma

        self.high_percentile = high_percentile
        self.low_ratio = low_ratio
        self.min_valid_response = min_valid_response

        self.weak_connection_dilate_size = weak_connection_dilate_size
        self.strong_connection_dilate_size = strong_connection_dilate_size

        self.min_expected_vessel_area_ratio = min_expected_vessel_area_ratio
        self.min_expected_skeleton_ratio = min_expected_skeleton_ratio

        self.remove_image_border_components = remove_image_border_components
        self.image_border_margin = image_border_margin

        self.use_skeleton = use_skeleton
        self.save_debug = save_debug

        self.validate_required_attributes()

        self.gabor_kernels = self.build_gabor_kernels()

        os.makedirs(self.output_mask_dir, exist_ok=True)
        os.makedirs(self.output_skeleton_dir, exist_ok=True)
        os.makedirs(self.debug_dir, exist_ok=True)

    def validate_required_attributes(self):
        required_attributes = [
            "frangi_weight",
            "gabor_weight",
            "product_keep_ratio",
            "response_cap_percentile",
            "response_median_ksize",
            "weak_connection_dilate_size",
            "strong_connection_dilate_size",
            "min_expected_vessel_area_ratio",
            "min_expected_skeleton_ratio",
        ]

        missing_attributes = []

        for attribute_name in required_attributes:
            if not hasattr(self, attribute_name):
                missing_attributes.append(attribute_name)

        if len(missing_attributes) > 0:
            raise AttributeError(
                "Missing required attributes in FundusVesselExtractor: "
                + ", ".join(missing_attributes)
            )

    def build_gabor_kernels(self):
        kernels = []

        if not self.use_gabor:
            return kernels

        for sigma in self.gabor_sigmas:
            for lambd in self.gabor_lambdas:
                for angle in range(0, 180, self.gabor_angle_step):
                    theta = np.deg2rad(angle)

                    kernel = cv2.getGaborKernel(
                        ksize=(self.gabor_ksize, self.gabor_ksize),
                        sigma=sigma,
                        theta=theta,
                        lambd=lambd,
                        gamma=self.gabor_gamma,
                        psi=0,
                        ktype=cv2.CV_32F
                    )

                    kernel = kernel - np.mean(kernel)

                    norm = np.sum(np.abs(kernel))

                    if norm > 1e-6:
                        kernel = kernel / norm

                    kernels.append(kernel)

        return kernels

    def get_base_image_id(self, green_file):
        image_id = os.path.splitext(green_file)[0]

        base_id = image_id.replace(
            "_green_preprocessed",
            ""
        )

        return base_id, image_id

    def load_fundus_mask(self, green_file, green):
        base_id, _ = self.get_base_image_id(green_file)

        mask_file = base_id + "_fundus_mask.png"

        mask_path = os.path.join(
            self.input_mask_dir,
            mask_file
        )

        if os.path.exists(mask_path):
            fundus_mask = cv2.imread(
                mask_path,
                cv2.IMREAD_GRAYSCALE
            )

            if fundus_mask is not None:
                if fundus_mask.shape != green.shape:
                    fundus_mask = cv2.resize(
                        fundus_mask,
                        (green.shape[1], green.shape[0]),
                        interpolation=cv2.INTER_NEAREST
                    )

                fundus_mask = (fundus_mask > 0).astype(np.uint8) * 255

                return fundus_mask, mask_path, "saved_mask"

        fundus_mask = self.create_fundus_mask_from_green(green)

        return fundus_mask, "", "recreated_from_green"

    def create_fundus_mask_from_green(self, green):
        _, mask = cv2.threshold(
            green,
            5,
            255,
            cv2.THRESH_BINARY
        )

        kernel = np.ones(
            (21, 21),
            np.uint8
        )

        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_CLOSE,
            kernel
        )

        contours, _ = cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        clean_mask = np.zeros_like(mask)

        if len(contours) > 0:
            largest_contour = max(
                contours,
                key=cv2.contourArea
            )

            cv2.drawContours(
                clean_mask,
                [largest_contour],
                -1,
                255,
                thickness=-1
            )
        else:
            clean_mask = mask.copy()

        clean_mask = cv2.erode(
            clean_mask,
            np.ones((7, 7), np.uint8),
            iterations=1
        )

        return clean_mask

    def resize_for_processing(self, green, fundus_mask):
        if self.process_scale >= 0.999:
            return green, fundus_mask

        new_w = max(
            1,
            int(green.shape[1] * self.process_scale)
        )

        new_h = max(
            1,
            int(green.shape[0] * self.process_scale)
        )

        green_small = cv2.resize(
            green,
            (new_w, new_h),
            interpolation=cv2.INTER_AREA
        )

        mask_small = cv2.resize(
            fundus_mask,
            (new_w, new_h),
            interpolation=cv2.INTER_NEAREST
        )

        mask_small = (mask_small > 0).astype(np.uint8) * 255

        return green_small, mask_small

    def resize_result_to_original(self, image, original_shape):
        if image.shape == original_shape:
            return image

        resized = cv2.resize(
            image,
            (original_shape[1], original_shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

        resized = (resized > 0).astype(np.uint8) * 255

        return resized

    def create_effective_mask(self, fundus_mask):
        if self.effective_mask_erode_size <= 1:
            effective_mask = fundus_mask.copy()
        else:
            kernel = np.ones(
                (
                    self.effective_mask_erode_size,
                    self.effective_mask_erode_size
                ),
                np.uint8
            )

            effective_mask = cv2.erode(
                fundus_mask,
                kernel,
                iterations=1
            )

        effective_mask = (effective_mask > 0).astype(np.uint8) * 255

        return effective_mask

    def normalize_inside_mask(self, image, mask):
        normalized = np.zeros_like(
            mask,
            dtype=np.uint8
        )

        image_f = image.astype(np.float32)

        pixels = image_f[mask > 0]

        if pixels.size == 0:
            return normalized

        p1 = np.percentile(
            pixels,
            1
        )

        p99 = np.percentile(
            pixels,
            99
        )

        if p99 <= p1:
            return normalized

        temp = (image_f - p1) / (p99 - p1)

        temp = np.clip(
            temp,
            0.0,
            1.0
        )

        temp = (temp * 255).astype(np.uint8)

        normalized = cv2.bitwise_and(
            temp,
            temp,
            mask=mask
        )

        return normalized

    def normalize_response_with_floor(
        self,
        response,
        mask,
        floor_percentile,
        cap_percentile=None
    ):
        if cap_percentile is None:
            cap_percentile = self.response_cap_percentile

        output = np.zeros_like(
            mask,
            dtype=np.uint8
        )

        pixels = response[mask > 0]

        if pixels.size == 0:
            return output

        floor_value = np.percentile(
            pixels,
            floor_percentile
        )

        cap_value = np.percentile(
            pixels,
            cap_percentile
        )

        if cap_value <= floor_value:
            return output

        temp = response.astype(np.float32)

        temp = (temp - floor_value) / (cap_value - floor_value)

        temp = np.clip(
            temp,
            0.0,
            1.0
        )

        temp = (temp * 255).astype(np.uint8)

        temp = cv2.bitwise_and(
            temp,
            temp,
            mask=mask
        )

        return temp

    def assess_image_quality(self, green, fundus_mask):
        pixels = green[fundus_mask > 0]

        if pixels.size == 0:
            return self.get_profile("normal")

        green_std = float(np.std(pixels))
        green_mean = float(np.mean(pixels))

        blurred = cv2.GaussianBlur(
            green,
            (5, 5),
            1.0
        )

        high_freq = cv2.absdiff(
            green,
            blurred
        )

        high_freq_std = float(
            np.std(high_freq[fundus_mask > 0])
        )

        lap = cv2.Laplacian(
            green,
            cv2.CV_32F,
            ksize=3
        )

        laplacian_std = float(
            np.std(lap[fundus_mask > 0])
        )

        if high_freq_std > 7.0 or laplacian_std > 18.0:
            profile = self.get_profile("noisy")
        elif green_std < 14.0:
            profile = self.get_profile("low_contrast")
        else:
            profile = self.get_profile("normal")

        profile["green_mean"] = green_mean
        profile["green_std"] = green_std
        profile["high_freq_std"] = high_freq_std
        profile["laplacian_std"] = laplacian_std

        return profile

    def get_profile(self, profile_name):
        if profile_name == "noisy":
            return {
                "profile_name": "noisy",

                "response_floor_percentile": 66,
                "frangi_floor_percentile": 62,
                "gabor_floor_percentile": 66,
                "dark_floor_percentile": 58,

                "high_percentile": 88.0,
                "low_ratio": 0.42,
                "min_valid_response": 5,

                "weak_frangi_ratio": 0.46,
                "weak_frangi_min_response": 4,

                "min_component_area": 18,
                "min_component_length": 8,
                "min_response_mean": 4.0,

                "min_score": 0.42,
                "connected_min_score": 0.28,

                "min_linearity": 0.08,
                "max_branch_density": 0.35,

                "min_aspect_ratio": 1.2,
                "max_component_extent": 0.96,

                "green_mean": 0.0,
                "green_std": 0.0,
                "high_freq_std": 0.0,
                "laplacian_std": 0.0,
            }

        if profile_name == "low_contrast":
            return {
                "profile_name": "low_contrast",

                "response_floor_percentile": 50,
                "frangi_floor_percentile": 48,
                "gabor_floor_percentile": 55,
                "dark_floor_percentile": 45,

                "high_percentile": 82.0,
                "low_ratio": 0.30,
                "min_valid_response": 2,

                "weak_frangi_ratio": 0.38,
                "weak_frangi_min_response": 2,

                "min_component_area": 10,
                "min_component_length": 5,
                "min_response_mean": 2.5,

                "min_score": 0.30,
                "connected_min_score": 0.18,

                "min_linearity": 0.05,
                "max_branch_density": 0.45,

                "min_aspect_ratio": 1.1,
                "max_component_extent": 0.98,

                "green_mean": 0.0,
                "green_std": 0.0,
                "high_freq_std": 0.0,
                "laplacian_std": 0.0,
            }

        return {
            "profile_name": "normal",

            "response_floor_percentile": 56,
            "frangi_floor_percentile": 54,
            "gabor_floor_percentile": 58,
            "dark_floor_percentile": 50,

            "high_percentile": 84.0,
            "low_ratio": 0.34,
            "min_valid_response": 3,

            "weak_frangi_ratio": 0.42,
            "weak_frangi_min_response": 3,

            "min_component_area": 14,
            "min_component_length": 7,
            "min_response_mean": 3.0,

            "min_score": 0.36,
            "connected_min_score": 0.22,

            "min_linearity": 0.06,
            "max_branch_density": 0.40,

            "min_aspect_ratio": 1.15,
            "max_component_extent": 0.97,

            "green_mean": 0.0,
            "green_std": 0.0,
            "high_freq_std": 0.0,
            "laplacian_std": 0.0,
        }

    def create_rescue_profile(self, profile):
        rescue = profile.copy()

        rescue["profile_name"] = profile["profile_name"] + "_rescue"

        rescue["response_floor_percentile"] = max(
            40,
            profile["response_floor_percentile"] - 10
        )

        rescue["frangi_floor_percentile"] = max(
            38,
            profile["frangi_floor_percentile"] - 10
        )

        rescue["gabor_floor_percentile"] = max(
            45,
            profile["gabor_floor_percentile"] - 8
        )

        rescue["dark_floor_percentile"] = max(
            40,
            profile["dark_floor_percentile"] - 8
        )

        rescue["high_percentile"] = max(
            76.0,
            profile["high_percentile"] - 6.0
        )

        rescue["low_ratio"] = max(
            0.25,
            profile["low_ratio"] - 0.06
        )

        rescue["min_valid_response"] = max(
            1,
            int(profile["min_valid_response"] * 0.6)
        )

        rescue["weak_frangi_ratio"] = max(
            0.30,
            profile["weak_frangi_ratio"] - 0.08
        )

        rescue["weak_frangi_min_response"] = max(
            1,
            int(profile["weak_frangi_min_response"] * 0.6)
        )

        rescue["min_component_area"] = max(
            5,
            int(profile["min_component_area"] * 0.55)
        )

        rescue["min_component_length"] = max(
            3,
            int(profile["min_component_length"] * 0.55)
        )

        rescue["min_response_mean"] = max(
            1.5,
            profile["min_response_mean"] * 0.6
        )

        rescue["min_score"] = max(
            0.18,
            profile["min_score"] - 0.12
        )

        rescue["connected_min_score"] = max(
            0.10,
            profile["connected_min_score"] - 0.08
        )

        rescue["min_linearity"] = max(
            0.03,
            profile["min_linearity"] * 0.6
        )

        rescue["max_branch_density"] = min(
            0.60,
            profile["max_branch_density"] + 0.12
        )

        return rescue

    def compute_frangi_response(self, green, fundus_mask, profile):
        green_norm = self.normalize_inside_mask(
            green,
            fundus_mask
        )

        img = green_norm.astype(np.float32)

        vesselness_max = np.zeros_like(
            img,
            dtype=np.float32
        )

        mask_bool = fundus_mask > 0

        for sigma in self.frangi_sigmas:
            ksize = int(6 * sigma + 1)

            if ksize % 2 == 0:
                ksize += 1

            smoothed = cv2.GaussianBlur(
                img,
                (ksize, ksize),
                sigma
            )

            ixx = cv2.Sobel(
                smoothed,
                cv2.CV_32F,
                2,
                0,
                ksize=3
            ) * (sigma ** 2)

            iyy = cv2.Sobel(
                smoothed,
                cv2.CV_32F,
                0,
                2,
                ksize=3
            ) * (sigma ** 2)

            ixy = cv2.Sobel(
                smoothed,
                cv2.CV_32F,
                1,
                1,
                ksize=3
            ) * (sigma ** 2)

            tmp = np.sqrt(
                np.maximum(
                    (ixx - iyy) ** 2 + 4.0 * (ixy ** 2),
                    0.0
                )
            )

            lambda1 = 0.5 * (ixx + iyy + tmp)
            lambda2 = 0.5 * (ixx + iyy - tmp)

            swap = np.abs(lambda1) > np.abs(lambda2)

            l1 = np.where(
                swap,
                lambda2,
                lambda1
            )

            l2 = np.where(
                swap,
                lambda1,
                lambda2
            )

            l2_safe = np.where(
                np.abs(l2) < 1e-6,
                1e-6,
                l2
            )

            rb = np.abs(l1) / np.abs(l2_safe)

            s = np.sqrt(
                l1 ** 2 + l2 ** 2
            )

            if np.any(mask_bool):
                s_max = float(np.max(s[mask_bool]))
            else:
                s_max = 0.0

            gamma = max(
                self.frangi_gamma_min,
                0.5 * s_max
            )

            vesselness = np.exp(
                -(rb ** 2) / (2.0 * self.frangi_beta ** 2)
            )

            vesselness = vesselness * (
                1.0 - np.exp(
                    -(s ** 2) / (2.0 * gamma ** 2)
                )
            )

            vesselness = np.where(
                l2 > 0,
                vesselness,
                0.0
            )

            vesselness_max = np.maximum(
                vesselness_max,
                vesselness
            )

        vesselness_max = np.where(
            mask_bool,
            vesselness_max,
            0.0
        )

        frangi_response = self.normalize_response_with_floor(
            vesselness_max,
            fundus_mask,
            floor_percentile=profile["frangi_floor_percentile"],
            cap_percentile=self.response_cap_percentile
        )

        return frangi_response, green_norm

    def compute_gabor_response(self, green_norm, fundus_mask, profile):
        if not self.use_gabor or len(self.gabor_kernels) == 0:
            return np.zeros_like(green_norm, dtype=np.uint8)

        inverted = 255 - green_norm

        inverted = cv2.bitwise_and(
            inverted,
            inverted,
            mask=fundus_mask
        )

        img = inverted.astype(np.float32)

        response_max = np.zeros_like(
            img,
            dtype=np.float32
        )

        for kernel in self.gabor_kernels:
            response = cv2.filter2D(
                img,
                cv2.CV_32F,
                kernel
            )

            response = np.maximum(
                response,
                0.0
            )

            response_max = np.maximum(
                response_max,
                response
            )

        gabor_response = self.normalize_response_with_floor(
            response_max,
            fundus_mask,
            floor_percentile=profile["gabor_floor_percentile"],
            cap_percentile=self.response_cap_percentile
        )

        return gabor_response

    def compute_dark_line_response(self, green_norm, fundus_mask, profile):
        local_background = cv2.GaussianBlur(
            green_norm,
            (0, 0),
            sigmaX=self.dark_background_sigma,
            sigmaY=self.dark_background_sigma
        )

        dark_response = cv2.subtract(
            local_background,
            green_norm
        )

        dark_response = cv2.bitwise_and(
            dark_response,
            dark_response,
            mask=fundus_mask
        )

        dark_response = self.normalize_response_with_floor(
            dark_response.astype(np.float32),
            fundus_mask,
            floor_percentile=profile["dark_floor_percentile"],
            cap_percentile=self.response_cap_percentile
        )

        return dark_response

    def combine_responses(
        self,
        frangi_response,
        gabor_response,
        dark_response,
        fundus_mask,
        profile
    ):
        frangi_f = frangi_response.astype(np.float32) / 255.0
        gabor_f = gabor_response.astype(np.float32) / 255.0
        dark_f = dark_response.astype(np.float32) / 255.0

        product_fg = np.sqrt(
            np.maximum(
                frangi_f * gabor_f,
                0.0
            )
        )

        weighted = (
            self.frangi_weight * frangi_f +
            self.gabor_weight * gabor_f +
            0.15 * dark_f
        )

        combined = np.maximum(
            product_fg,
            self.product_keep_ratio * weighted
        )

        # Frangi単独でもある程度残す。
        # Gaborに合わない薄い血管を消しすぎないため。
        combined = np.maximum(
            combined,
            0.55 * frangi_f
        )

        combined = np.clip(
            combined,
            0.0,
            1.0
        )

        combined = (combined * 255).astype(np.uint8)

        combined = cv2.bitwise_and(
            combined,
            combined,
            mask=fundus_mask
        )

        if self.response_median_ksize >= 3:
            ksize = self.response_median_ksize

            if ksize % 2 == 0:
                ksize += 1

            combined = cv2.medianBlur(
                combined,
                ksize
            )

        combined = cv2.GaussianBlur(
            combined,
            (3, 3),
            0
        )

        combined = self.normalize_response_with_floor(
            combined.astype(np.float32),
            fundus_mask,
            floor_percentile=profile["response_floor_percentile"],
            cap_percentile=self.response_cap_percentile
        )

        return combined

    def extract_vessel_response(self, green, fundus_mask, profile):
        frangi_response, green_norm = self.compute_frangi_response(
            green,
            fundus_mask,
            profile
        )

        gabor_response = self.compute_gabor_response(
            green_norm,
            fundus_mask,
            profile
        )

        dark_response = self.compute_dark_line_response(
            green_norm,
            fundus_mask,
            profile
        )

        vessel_response = self.combine_responses(
            frangi_response,
            gabor_response,
            dark_response,
            fundus_mask,
            profile
        )

        return (
            vessel_response,
            frangi_response,
            gabor_response,
            dark_response,
            green_norm
        )

    def otsu_threshold_from_masked_pixels(self, response, fundus_mask):
        pixels = response[fundus_mask > 0].astype(np.uint8)

        if pixels.size == 0:
            return 0.0

        pixels_2d = pixels.reshape(-1, 1)

        otsu_value, _ = cv2.threshold(
            pixels_2d,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        return float(otsu_value)

    def hysteresis_threshold(
        self,
        response,
        fundus_mask,
        frangi_response,
        dark_response,
        profile
    ):
        valid = (
            (fundus_mask > 0) &
            (response >= profile["min_valid_response"])
        )

        pixels = response[valid]

        if pixels.size == 0:
            empty = np.zeros_like(response, dtype=np.uint8)
            return empty, empty, 0.0, 0.0

        high_threshold = float(
            np.percentile(
                pixels,
                profile["high_percentile"]
            )
        )

        otsu_threshold = self.otsu_threshold_from_masked_pixels(
            response,
            fundus_mask
        )

        if otsu_threshold > 0:
            high_threshold = min(
                high_threshold,
                otsu_threshold * 1.20
            )

        high_threshold = max(
            high_threshold,
            profile["min_valid_response"]
        )

        low_threshold = max(
            high_threshold * profile["low_ratio"],
            profile["min_valid_response"] * 0.75
        )

        strong = np.zeros_like(response, dtype=np.uint8)
        weak = np.zeros_like(response, dtype=np.uint8)

        strong[
            (response >= high_threshold) &
            (fundus_mask > 0)
        ] = 255

        weak[
            (response >= low_threshold) &
            (fundus_mask > 0)
        ] = 255

        frangi_valid = (
            (fundus_mask > 0) &
            (frangi_response >= profile["weak_frangi_min_response"])
        )

        frangi_pixels = frangi_response[frangi_valid]

        if frangi_pixels.size > 0:
            frangi_high = float(
                np.percentile(
                    frangi_pixels,
                    profile["high_percentile"]
                )
            )

            frangi_weak_threshold = frangi_high * profile["weak_frangi_ratio"]

            frangi_weak = np.zeros_like(response, dtype=np.uint8)

            frangi_weak[
                (frangi_response >= frangi_weak_threshold) &
                (fundus_mask > 0)
            ] = 255

            weak = cv2.bitwise_or(
                weak,
                frangi_weak
            )

        dark_pixels = dark_response[fundus_mask > 0]

        if dark_pixels.size > 0:
            dark_threshold = np.percentile(
                dark_pixels,
                70
            ) * 0.35

            dark_weak = np.zeros_like(response, dtype=np.uint8)

            dark_weak[
                (dark_response >= dark_threshold) &
                (fundus_mask > 0)
            ] = 255

            weak = cv2.bitwise_or(
                weak,
                dark_weak
            )

        close_size = 3

        if profile["profile_name"].startswith("low_contrast"):
            close_size = 5

        if close_size >= 3:
            close_kernel = cv2.getStructuringElement(
                cv2.MORPH_ELLIPSE,
                (close_size, close_size)
            )

            weak = cv2.morphologyEx(
                weak,
                cv2.MORPH_CLOSE,
                close_kernel
            )

            weak = cv2.bitwise_and(
                weak,
                weak,
                mask=fundus_mask
            )

        connection_size = self.weak_connection_dilate_size

        if profile["profile_name"].startswith("low_contrast"):
            connection_size = max(
                connection_size,
                13
            )

        if connection_size % 2 == 0:
            connection_size += 1

        connection_kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (connection_size, connection_size)
        )

        strong_neighborhood = cv2.dilate(
            strong,
            connection_kernel,
            iterations=1
        )

        num_labels, labels = cv2.connectedComponents(
            weak,
            connectivity=8
        )

        result = np.zeros_like(response, dtype=np.uint8)

        for label_id in range(1, num_labels):
            component = labels == label_id

            touches_strong = np.any(
                strong_neighborhood[component] > 0
            )

            if touches_strong:
                result[component] = 255

        if np.count_nonzero(result) < 20:
            result = weak.copy()

        return result, strong, float(high_threshold), float(low_threshold)

    def skeletonize_binary(self, binary):
        binary = (binary > 0).astype(np.uint8) * 255

        if hasattr(cv2, "ximgproc"):
            skeleton = cv2.ximgproc.thinning(
                binary,
                thinningType=cv2.ximgproc.THINNING_ZHANGSUEN
            )

            return skeleton

        skeleton = np.zeros_like(binary)

        element = cv2.getStructuringElement(
            cv2.MORPH_CROSS,
            (3, 3)
        )

        temp = binary.copy()

        while True:
            opened = cv2.morphologyEx(
                temp,
                cv2.MORPH_OPEN,
                element
            )

            subtracted = cv2.subtract(
                temp,
                opened
            )

            eroded = cv2.erode(
                temp,
                element
            )

            skeleton = cv2.bitwise_or(
                skeleton,
                subtracted
            )

            temp = eroded.copy()

            if cv2.countNonZero(temp) == 0:
                break

        return skeleton

    def count_branch_points(self, skeleton):
        skel = (skeleton > 0).astype(np.uint8)

        kernel = np.ones(
            (3, 3),
            dtype=np.float32
        )

        kernel[1, 1] = 0.0

        neighbor_count = cv2.filter2D(
            skel.astype(np.float32),
            cv2.CV_32F,
            kernel,
            borderType=cv2.BORDER_CONSTANT
        )

        branch_points = np.count_nonzero(
            (skel > 0) &
            (neighbor_count >= 3.0)
        )

        return int(branch_points)

    def compute_component_shape_metrics(self, component_image):
        skeleton = self.skeletonize_binary(
            component_image
        )

        area = np.count_nonzero(
            component_image
        )

        skeleton_length = np.count_nonzero(
            skeleton
        )

        if area == 0 or skeleton_length == 0:
            return {
                "skeleton": skeleton,
                "linearity": 0.0,
                "branch_density": 1.0,
                "skeleton_length": 0,
                "branch_points": 0,
            }

        branch_points = self.count_branch_points(
            skeleton
        )

        linearity = skeleton_length / max(
            float(area),
            1.0
        )

        branch_density = branch_points / max(
            float(skeleton_length),
            1.0
        )

        return {
            "skeleton": skeleton,
            "linearity": float(linearity),
            "branch_density": float(branch_density),
            "skeleton_length": int(skeleton_length),
            "branch_points": int(branch_points),
        }

    def remove_components_by_score(
        self,
        binary,
        response,
        dark_response,
        strong_seed,
        profile
    ):
        binary = (binary > 0).astype(np.uint8) * 255

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            binary,
            connectivity=8
        )

        cleaned = np.zeros_like(binary)

        rejection_counts = {
            "border": 0,
            "area": 0,
            "length": 0,
            "score": 0,
            "accepted": 0,
        }

        h, w = binary.shape

        dilate_kernel = np.ones(
            (
                self.strong_connection_dilate_size,
                self.strong_connection_dilate_size
            ),
            np.uint8
        )

        strong_dilated = cv2.dilate(
            strong_seed,
            dilate_kernel,
            iterations=1
        )

        for label_id in range(1, num_labels):
            area = stats[label_id, cv2.CC_STAT_AREA]
            x = stats[label_id, cv2.CC_STAT_LEFT]
            y = stats[label_id, cv2.CC_STAT_TOP]
            width = stats[label_id, cv2.CC_STAT_WIDTH]
            height = stats[label_id, cv2.CC_STAT_HEIGHT]

            bbox_length = max(width, height)
            bbox_short = max(1, min(width, height))
            aspect_ratio = bbox_length / bbox_short
            bbox_area = max(1, width * height)
            extent = area / bbox_area

            component_mask = labels == label_id

            if self.remove_image_border_components:
                touches_image_border = (
                    x <= self.image_border_margin or
                    y <= self.image_border_margin or
                    x + width >= w - self.image_border_margin or
                    y + height >= h - self.image_border_margin
                )

                if touches_image_border:
                    rejection_counts["border"] += 1
                    continue

            connected_to_strong = bool(
                np.any(strong_dilated[component_mask] > 0)
            )

            min_area = profile["min_component_area"]
            min_length = profile["min_component_length"]

            if connected_to_strong:
                min_area = max(
                    1,
                    int(min_area * 0.45)
                )

                min_length = max(
                    1,
                    int(min_length * 0.45)
                )

            if area < min_area:
                rejection_counts["area"] += 1
                continue

            if bbox_length < min_length:
                rejection_counts["length"] += 1
                continue

            component_response_mean = float(
                np.mean(response[component_mask])
            )

            component_dark_mean = float(
                np.mean(dark_response[component_mask])
            )

            component_image = np.zeros_like(binary)
            component_image[component_mask] = 255

            shape_metrics = self.compute_component_shape_metrics(
                component_image
            )

            linearity = shape_metrics["linearity"]
            branch_density = shape_metrics["branch_density"]

            response_score = min(
                component_response_mean / 80.0,
                1.0
            )

            dark_score = min(
                component_dark_mean / 80.0,
                1.0
            )

            linearity_score = min(
                linearity / 0.20,
                1.0
            )

            branch_penalty = min(
                branch_density / 0.50,
                1.0
            )

            aspect_score = min(
                aspect_ratio / 3.0,
                1.0
            )

            compact_penalty = 0.0

            if extent > profile["max_component_extent"] and area < profile["min_component_area"] * 4:
                compact_penalty = 0.25

            score = (
                0.35 * response_score +
                0.15 * dark_score +
                0.25 * linearity_score +
                0.15 * aspect_score +
                0.20 * (1.0 - branch_penalty)
            )

            score = score - compact_penalty

            required_score = profile["min_score"]

            if connected_to_strong:
                required_score = profile["connected_min_score"]

            if score < required_score:
                rejection_counts["score"] += 1
                continue

            cleaned[component_mask] = 255
            rejection_counts["accepted"] += 1

        return cleaned, rejection_counts

    def morphological_skeleton(self, binary):
        binary = (binary > 0).astype(np.uint8) * 255

        if not self.use_skeleton:
            return binary

        return self.skeletonize_binary(
            binary
        )

    def vessel_distribution_score(self, vessel_binary):
        binary = vessel_binary > 0

        h, w = binary.shape

        cell_h = h // self.grid_size
        cell_w = w // self.grid_size

        if cell_h == 0 or cell_w == 0:
            return 0.0

        occupied_cells = 0
        total_cells = self.grid_size * self.grid_size

        for gy in range(self.grid_size):
            for gx in range(self.grid_size):
                y0 = gy * cell_h
                y1 = h if gy == self.grid_size - 1 else (gy + 1) * cell_h

                x0 = gx * cell_w
                x1 = w if gx == self.grid_size - 1 else (gx + 1) * cell_w

                cell = binary[y0:y1, x0:x1]

                if np.any(cell):
                    occupied_cells += 1

        score = occupied_cells / total_cells

        return score

    def calculate_vessel_metrics(self, vessel_mask, vessel_skeleton, fundus_mask):
        fundus_area = np.count_nonzero(
            fundus_mask
        )

        if fundus_area == 0:
            return {
                "vessel_area_ratio": 0.0,
                "skeleton_length_ratio": 0.0,
                "distribution_score": 0.0,
            }

        vessel_area = np.count_nonzero(
            vessel_mask
        )

        skeleton_length = np.count_nonzero(
            vessel_skeleton
        )

        return {
            "vessel_area_ratio": float(vessel_area / fundus_area),
            "skeleton_length_ratio": float(skeleton_length / fundus_area),
            "distribution_score": float(
                self.vessel_distribution_score(vessel_skeleton)
            ),
        }

    def extract_once_with_profile(
        self,
        green,
        effective_mask,
        profile
    ):
        (
            vessel_response,
            frangi_response,
            gabor_response,
            dark_response,
            green_norm
        ) = self.extract_vessel_response(
            green,
            effective_mask,
            profile
        )

        vessel_mask_raw, strong_seed, high_threshold, low_threshold = self.hysteresis_threshold(
            vessel_response,
            effective_mask,
            frangi_response=frangi_response,
            dark_response=dark_response,
            profile=profile
        )

        vessel_mask_cleaned, rejection_counts = self.remove_components_by_score(
            vessel_mask_raw,
            vessel_response,
            dark_response,
            strong_seed=strong_seed,
            profile=profile
        )

        vessel_mask_cleaned = cv2.bitwise_and(
            vessel_mask_cleaned,
            vessel_mask_cleaned,
            mask=effective_mask
        )

        return {
            "vessel_mask": vessel_mask_cleaned,
            "vessel_response": vessel_response,
            "frangi_response": frangi_response,
            "gabor_response": gabor_response,
            "dark_response": dark_response,
            "green_norm": green_norm,
            "strong_seed": strong_seed,
            "vessel_mask_raw": vessel_mask_raw,
            "high_threshold": high_threshold,
            "low_threshold": low_threshold,
            "rejection_counts": rejection_counts,
            "profile": profile,
        }

    def should_rescue(self, vessel_mask, vessel_skeleton, fundus_mask):
        fundus_area = np.count_nonzero(fundus_mask)

        if fundus_area == 0:
            return False

        vessel_area_ratio = np.count_nonzero(vessel_mask) / fundus_area
        skeleton_ratio = np.count_nonzero(vessel_skeleton) / fundus_area

        if vessel_area_ratio < self.min_expected_vessel_area_ratio:
            return True

        if skeleton_ratio < self.min_expected_skeleton_ratio:
            return True

        return False

    def extract_one_image(self, green_original, fundus_mask_original):
        original_shape = green_original.shape

        green, fundus_mask = self.resize_for_processing(
            green_original,
            fundus_mask_original
        )

        effective_mask = self.create_effective_mask(
            fundus_mask
        )

        profile = self.assess_image_quality(
            green,
            effective_mask
        )

        result = self.extract_once_with_profile(
            green,
            effective_mask,
            profile
        )

        vessel_skeleton = self.morphological_skeleton(
            result["vessel_mask"]
        )

        recovery_used = False

        if self.should_rescue(
            result["vessel_mask"],
            vessel_skeleton,
            effective_mask
        ):
            rescue_profile = self.create_rescue_profile(
                profile
            )

            rescue_result = self.extract_once_with_profile(
                green,
                effective_mask,
                rescue_profile
            )

            rescue_skeleton = self.morphological_skeleton(
                rescue_result["vessel_mask"]
            )

            if np.count_nonzero(rescue_result["vessel_mask"]) > np.count_nonzero(result["vessel_mask"]):
                result = rescue_result
                vessel_skeleton = rescue_skeleton
                profile = rescue_profile
                recovery_used = True

        vessel_mask_original = self.resize_result_to_original(
            result["vessel_mask"],
            original_shape
        )

        vessel_skeleton_original = self.resize_result_to_original(
            vessel_skeleton,
            original_shape
        )

        effective_mask_original = self.resize_result_to_original(
            effective_mask,
            original_shape
        )

        metrics = self.calculate_vessel_metrics(
            vessel_mask_original,
            vessel_skeleton_original,
            effective_mask_original
        )

        metrics["process_scale"] = self.process_scale
        metrics["profile_name"] = profile["profile_name"]
        metrics["green_mean"] = profile["green_mean"]
        metrics["green_std"] = profile["green_std"]
        metrics["high_freq_std"] = profile["high_freq_std"]
        metrics["laplacian_std"] = profile["laplacian_std"]
        metrics["high_threshold"] = result["high_threshold"]
        metrics["low_threshold"] = result["low_threshold"]
        metrics["recovery_used"] = recovery_used

        for key, value in result["rejection_counts"].items():
            metrics["reject_" + key] = value

        steps = {}

        if self.save_debug:
            steps["01_green_input"] = green.copy()
            steps["02_fundus_mask_effective"] = effective_mask.copy()
            steps["03_green_normalized"] = result["green_norm"].copy()
            steps["04_frangi_response"] = result["frangi_response"].copy()
            steps["05_gabor_response"] = result["gabor_response"].copy()
            steps["06_dark_response"] = result["dark_response"].copy()
            steps["07_combined_response"] = result["vessel_response"].copy()
            steps["08_strong_seed"] = result["strong_seed"].copy()
            steps["09_vessel_mask_raw"] = result["vessel_mask_raw"].copy()
            steps["10_vessel_mask_cleaned"] = result["vessel_mask"].copy()
            steps["11_vessel_skeleton"] = vessel_skeleton.copy()

        return vessel_mask_original, vessel_skeleton_original, steps, metrics

    def save_debug_steps(self, steps, save_dir):
        os.makedirs(
            save_dir,
            exist_ok=True
        )

        for name, image in steps.items():
            save_path = os.path.join(
                save_dir,
                name + ".png"
            )

            cv2.imwrite(
                save_path,
                image
            )

    def run(self):
        files = sorted([
            file for file in os.listdir(self.input_green_dir)
            if file.lower().endswith(".png")
        ])

        if len(files) == 0:
            raise FileNotFoundError(
                f"{self.input_green_dir} にpng画像がありません。"
            )

        random.seed(
            self.random_seed
        )

        debug_files = set(
            random.sample(
                files,
                min(
                    self.debug_sample_count,
                    len(files)
                )
            )
        )

        records = []
        debug_index = 1

        for file in tqdm(files):
            image_path = os.path.join(
                self.input_green_dir,
                file
            )

            green = cv2.imread(
                image_path,
                cv2.IMREAD_GRAYSCALE
            )

            if green is None:
                print(f"画像を読み込めませんでした: {image_path}")
                continue

            fundus_mask, fundus_mask_path, mask_source = self.load_fundus_mask(
                file,
                green
            )

            vessel_mask, vessel_skeleton, steps, metrics = self.extract_one_image(
                green,
                fundus_mask
            )

            base_id, image_id = self.get_base_image_id(
                file
            )

            mask_output_path = os.path.join(
                self.output_mask_dir,
                image_id + "_vessel_mask.png"
            )

            skeleton_output_path = os.path.join(
                self.output_skeleton_dir,
                image_id + "_vessel_skeleton.png"
            )

            cv2.imwrite(
                mask_output_path,
                vessel_mask
            )

            cv2.imwrite(
                skeleton_output_path,
                vessel_skeleton
            )

            record = {
                "image": file,
                "base_id": base_id,
                "profile_name": metrics["profile_name"],
                "recovery_used": metrics["recovery_used"],
                "green_mean": metrics["green_mean"],
                "green_std": metrics["green_std"],
                "high_freq_std": metrics["high_freq_std"],
                "laplacian_std": metrics["laplacian_std"],
                "fundus_mask_path": fundus_mask_path,
                "mask_source": mask_source,
                "vessel_mask_output": mask_output_path,
                "skeleton_output": skeleton_output_path,
                "vessel_area_ratio": metrics["vessel_area_ratio"],
                "skeleton_length_ratio": metrics["skeleton_length_ratio"],
                "distribution_score": metrics["distribution_score"],
                "process_scale": metrics["process_scale"],
                "high_threshold": metrics["high_threshold"],
                "low_threshold": metrics["low_threshold"],
                "effective_mask_erode_size": self.effective_mask_erode_size,
                "frangi_sigmas": str(self.frangi_sigmas),
                "frangi_beta": self.frangi_beta,
                "frangi_gamma_min": self.frangi_gamma_min,
                "use_gabor": self.use_gabor,
                "gabor_ksize": self.gabor_ksize,
                "gabor_sigmas": str(self.gabor_sigmas),
                "gabor_lambdas": str(self.gabor_lambdas),
                "gabor_gamma": self.gabor_gamma,
                "gabor_angle_step": self.gabor_angle_step,
                "frangi_weight": self.frangi_weight,
                "gabor_weight": self.gabor_weight,
                "product_keep_ratio": self.product_keep_ratio,
                "response_cap_percentile": self.response_cap_percentile,
                "response_median_ksize": self.response_median_ksize,
            }

            for key, value in metrics.items():
                if key.startswith("reject_"):
                    record[key] = value

            records.append(record)

            if self.save_debug and file in debug_files:
                sample_dir = os.path.join(
                    self.debug_dir,
                    f"sample{debug_index:02d}_{image_id}"
                )

                self.save_debug_steps(
                    steps,
                    sample_dir
                )

                debug_index += 1

        df = pd.DataFrame(records)

        df.to_csv(
            self.metrics_csv,
            index=False
        )

        print("血管抽出が完了しました。")
        print(f"入力G画像: {self.input_green_dir}")
        print(f"入力眼底領域マスク: {self.input_mask_dir}")
        print(f"血管マスク画像: {self.output_mask_dir}")
        print(f"血管スケルトン画像: {self.output_skeleton_dir}")
        print(f"デバッグ画像: {self.debug_dir}")
        print(f"血管抽出指標CSV: {self.metrics_csv}")


if __name__ == "__main__":
    extractor = FundusVesselExtractor(
        input_green_dir="preprocessed_green_images",
        input_mask_dir="fundus_masks",
        output_mask_dir="vessel_masks",
        output_skeleton_dir="vessel_skeletons",
        debug_dir="vessel_debug_samples",
        metrics_csv="vessel_extraction_metrics.csv",

        debug_sample_count=1,
        random_seed=42,
        grid_size=8,

        process_scale=0.85,
        effective_mask_erode_size=15,

        frangi_sigmas=(0.8, 1.0, 1.4, 2.0, 2.8),
        frangi_beta=0.40,
        frangi_gamma_min=2.0,

        use_gabor=True,
        gabor_ksize=17,
        gabor_sigmas=(2.0, 3.0),
        gabor_lambdas=(8.0, 12.0),
        gabor_gamma=0.45,
        gabor_angle_step=30,

        frangi_weight=0.65,
        gabor_weight=0.35,
        product_keep_ratio=0.45,

        response_cap_percentile=99.5,
        response_median_ksize=3,

        dark_background_sigma=9,

        high_percentile=86.0,
        low_ratio=0.38,
        min_valid_response=3,

        weak_connection_dilate_size=11,
        strong_connection_dilate_size=7,

        min_expected_vessel_area_ratio=0.002,
        min_expected_skeleton_ratio=0.0005,

        remove_image_border_components=True,
        image_border_margin=4,

        use_skeleton=True,
        save_debug=True,
    )

    extractor.run()

100%|██████████| 1434/1434 [1:58:36<00:00,  4.96s/it]  

血管抽出が完了しました。
入力G画像: preprocessed_green_images
入力眼底領域マスク: fundus_masks
血管マスク画像: vessel_masks
血管スケルトン画像: vessel_skeletons
デバッグ画像: vessel_debug_samples
血管抽出指標CSV: vessel_extraction_metrics.csv
